<a href="https://colab.research.google.com/github/yashita-mishra/AI-Project-Child-Nutrition/blob/main/Nhanescleanup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import urllib.request

import numpy as np
import pandas as pd

BASE = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/"
DATA_DIR = "data"

KEEP = {
    "DEMO_L": [
        "SEQN", "RIDSTATR", "RIAGENDR", "RIDAGEYR", "RIDAGEMN", "RIDEXAGM",
        "RIDRETH3", "DMDBORN4", "DMDHHSIZ", "DMDHRGND", "DMDHRAGZ", "DMDHREDZ",
        "DMDHSEDZ", "DMDHRMAZ", "INDFMPIR", "WTINT2YR", "WTMEC2YR",
        "SDMVSTRA", "SDMVPSU",
    ],
    "BMX_L": ["SEQN", "BMXWT", "BMXHT", "BMXBMI", "BMXRECUM"],
    "CBC_L": ["SEQN", "LBXHGB", "LBXHCT"],
    "ECQ_L": ["SEQN", "ECD070A", "ECD070B", "ECQ080", "ECQ090", "WHQ030E", "MCQ080E"],
    "DBQ_L": ["SEQN", "DBQ010", "DBD030", "DBD041", "DBD050", "DBD055", "DBD061"],
    "FSQ_L": [
        "SEQN", "FSDAD", "FSD162", "FSQ012", "FSQ165", "FSQ653", "FSD660ZC",
        "FSD675", "FSD680", "FSD670ZC", "FSQ690",
    ],
}

MISSING_CODES = {
    **{c: [7, 9] for c in [
        "DBQ010", "ECQ080", "ECQ090", "WHQ030E", "MCQ080E", "FSD162", "FSQ012",
        "FSQ165", "FSQ653", "FSD660ZC", "FSD675", "FSD680", "FSQ690",
    ]},
    "FSD670ZC": [777, 999],
    "ECD070A": [7777, 9999],
    "ECD070B": [7777, 9999],
    **{c: [777777, 999999] for c in ["DBD030", "DBD041", "DBD050", "DBD055", "DBD061"]},
}

CENSORED = {"DBD030": 1096, "DBD041": 366, "DBD050": 1096, "DBD055": 731, "DBD061": 1096}

ZERO_MEANS = {
    "DBD030": "still_breastfeeding",
    "DBD041": "formula_never",
    "DBD050": "still_formula",
    "DBD055": "other_food_never",
    "DBD061": "milk_never",
}


def fetch(name: str) -> pd.DataFrame:
    path = os.path.join(DATA_DIR, f"{name}.xpt")
    if not os.path.exists(path):
        os.makedirs(DATA_DIR, exist_ok=True)
        print(f"downloading {name} ...")
        req = urllib.request.Request(
            BASE + name + ".xpt", headers={"User-Agent": "Mozilla/5.0"}
        )
        with urllib.request.urlopen(req) as r, open(path, "wb") as f:
            f.write(r.read())
    df = pd.read_sas(path, format="xport")
    keep = [c for c in KEEP[name] if c in df.columns]
    absent = sorted(set(KEEP[name]) - set(keep))
    if absent:
        print(f"[{name}] expected columns not found: {absent}")
    return df[keep]

In [2]:
frames = {name: fetch(name) for name in KEEP}

df = frames["DEMO_L"]
n_demo = len(df)
for name, part in frames.items():
    if name == "DEMO_L":
        continue
    df = df.merge(part, on="SEQN", how="left", validate="one_to_one")
    assert len(df) == n_demo, f"{name} changed the row count"
print(f"merged table: {len(df)} rows x {df.shape[1]} columns")

# refused / don't know -> NaN
for col, codes in MISSING_CODES.items():
    if col in df:
        df[col] = df[col].replace(codes, np.nan)

# censored ages: flag, then replace by the cap
for col, cap in CENSORED.items():
    if col in df:
        df[f"{col}_censored"] = df[col] == 666666
        df[col] = df[col].where(df[col] != 666666, cap)

# zeros meaning "still" / "never": explicit flags, blank the age
for col, flag in ZERO_MEANS.items():
    if col in df:
        df[flag] = df[col] == 0
        df[col] = df[col].replace(0, np.nan)

kids = df[df["RIDAGEYR"].between(2, 6) & (df["RIDSTATR"] == 2)].copy()
print(f"children aged 2-6 who were examined: {len(kids)}")

downloading DEMO_L ...
downloading BMX_L ...
downloading CBC_L ...
downloading ECQ_L ...
downloading DBQ_L ...
downloading FSQ_L ...
merged table: 11933 rows x 47 columns
children aged 2-6 who were examined: 664


In [3]:
if {"ECD070A", "ECD070B"} <= set(kids.columns):
    kids["birth_wt_lb"] = kids["ECD070A"] + kids["ECD070B"] / 16
    lbw = pd.Series(
        np.where(kids["birth_wt_lb"].notna(), kids["birth_wt_lb"] < 5.5, np.nan),
        index=kids.index,
    )
    if "ECQ080" in kids:
        lbw = lbw.fillna(kids["ECQ080"].map({1: 0.0, 2: 1.0}))
    kids["low_birth_weight"] = lbw

if {"DMDHRGND", "DMDHREDZ", "DMDHSEDZ"} <= set(kids.columns):
    kids["parent_edu"] = np.where(
        kids["DMDHRGND"] == 2, kids["DMDHREDZ"], kids["DMDHSEDZ"]
    )

if "LBXHGB" in kids:
    cutoff = np.where(kids["RIDAGEYR"] < 5, 11.0, 11.5)
    kids["anaemia"] = np.where(
        kids["LBXHGB"].notna(), (kids["LBXHGB"] < cutoff).astype(float), np.nan
    )

kids.to_csv(os.path.join(DATA_DIR, "children_2_6.csv"), index=False)
kids.isna().mean().sort_values(ascending=False).round(2).head(25)

,0
RIDAGEMN,0.99
ECQ090,0.98
ECQ080,0.98
BMXRECUM,0.69
FSD680,0.60
FSD670ZC,0.55
FSD675,0.55
FSD660ZC,0.55
anaemia,0.49
LBXHCT,0.49


In [4]:
from google.colab import files
files.download("data/children_2_6.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>